# Carbon Chain Workflow
This notebook will demonstrate a self consistent workflow on a carbon chain.

In [1]:
from BigDFT.Systems import System
from BigDFT.Fragments import Fragment
from BigDFT.Atoms import Atom
from BigDFT.UnitCells import UnitCell
from matplotlib import pyplot as plt

Define the size in terms of the middle, left, and right.

In [2]:
# size = [8, 8, 8]
size = [6, 4, 4]
spacing = 1.4
cell = UnitCell(["inf", "inf", spacing * (size[0] + size[1] + size[2])], units="angstroem")
name = "-".join([str(x) for x in size])

## System Construction
Now the system can be built.

In [3]:
sys = System()
sys.cell = cell
idx = 0

sys["LEF:2"] = Fragment()
for i in range(size[1] // 2):
    at = Atom({"C": [0, 0, idx * spacing], "units": "angstroem"})
    sys["LEF:2"].append(at)
    idx += 1
sys["LEF:1"] = Fragment()
for i in range(size[1] // 2):
    at = Atom({"C": [0, 0, idx * spacing], "units": "angstroem"})
    sys["LEF:1"].append(at)
    idx += 1
sys["MID:0"] = Fragment()
for i in range(size[0]):
    at = Atom({"C": [0, 0, idx * spacing], "units": "angstroem"})
    sys["MID:0"].append(at)
    idx += 1
sys["RIG:1"] = Fragment()
for i in range(size[2] // 2):
    at = Atom({"C": [0, 0, idx * spacing], "units": "angstroem"})
    sys["RIG:1"].append(at)
    idx += 1
sys["RIG:2"] = Fragment()
for i in range(size[2] // 2):
    at = Atom({"C": [0, 0, idx * spacing], "units": "angstroem"})
    sys["RIG:2"].append(at)
    idx += 1

Make a pdf file for visualization.

In [4]:
from BigDFT.IO import write_pdb
with open("test.pdb", "w") as ofile:
    write_pdb(sys, ofile)

In [5]:
from BigDFT.Visualization import InlineVisualizer
viz=InlineVisualizer(600, 500) #---> activate visualizer ---> jupyter-notebook
viz.display_system(sys)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Calculation Setup
Now we need a function that can run the BigDFT calculation for the base ground state.

In [6]:
from BigDFT.Calculators import SystemCalculator
calc = SystemCalculator(mpi_run="mpirun -np 2", omp=2, verbose=False)

In [7]:
def make_input(libnegf=False, regions=None, reorder=None):
    from BigDFT.Inputfiles import Inputfile
    inp = Inputfile()
    inp.set_psp_nlcc()
    inp.set_xc("PBE")
    inp.set_hgrid(0.5)
    inp["import"] = "linear"
    if libnegf:
        inp["chess"] = {"libnegf": {"regions": regions, "reorder": reorder}, "foe": {"occupation_function": 106}}
        inp["lin_kernel"] = {"libnegf_on": True}
        inp["dft"]["inputpsiid"] = "linear_restart"
        inp["perf"] = {"adjust_kernel_iterations": False, "adjust_kernel_threshold": False}
        inp["lin_basis"] = {"idsx": 0, "nit": 0}
        inp["lin_general"] = {"hybrid": False, "kernel_restart_mode": "kernel"}
    else:
        inp.write_support_functions_on_disk()
    return inp

Run the first calculation.

In [9]:
logs = {}
inp = make_input()
logs["GS"] = calc.run(sys=sys, input=inp, name=name, run_dir="scr", skip=True)

TypeError: argument of type 'NoneType' is not iterable

Plot convergence.

In [ ]:
from matplotlib import pyplot as plt
fig, axs = plt.subplots(1, 1, figsize=(6, 3))
axs.plot(logs["GS"].get_intermediate_energies(), 'o--')

Copy it so we can start working on the restart runs.

In [ ]:
from os import system
system(f"rsync -uavq scr/ scr2/")

## LibNEGF Postprocessing
We can do a single shot of libNEGF postprocessing using the driver included in this folder. To compile it myself I had to do:
```
mpif90 test_negf_driver.f90 -o test_driv \
-I ../bigdft-suite/build2/install/include/ \
-L ../bigdft-suite/build2/install/lib -lfutile-1 -lyaml -lNTPoly -lCheSS-1 \
-L ../lib -lnegf -lmpifx \
 -framework Accelerate  -fopenmp
```
Update the directories according to where you built bigdft and libNEGF. 

First we prepare information about the order of basis functions for each region.

In [9]:
def get_ordering(sys, log, tool):
    """
    Get the indices we will use to match LibNEGF's internal ordering.

    Args:
        sys (BigDFT.Systems.System): the system being studied. It must be
        setup into the right set of fragments (MID:0, LEF:1, LEF:2, RIG:1,
        RIG:2).
        log (BigDFT.Logfiles.Logfile): the log of a calculation.
        tool (BigDFT.PostProcessing.BigDFTool): tool for post processing.

    Returns:
        order (list): the listed order of indices.
        idx (dict): the fragment indices.
    """
    idx = tool.get_frag_indices(sys, log)
    order = idx["MID:0"] + idx["LEF:1"] + idx["LEF:2"] + idx["RIG:1"] + idx["RIG:2"]
    return order, idx

In [10]:
from BigDFT.PostProcessing import BigDFTool
tool = BigDFTool()
order, idx = get_ordering(sys, logs["GS"], tool)
lengths = [len(idx["MID:0"])]
for key in ["LEF:1", "LEF:2", "RIG:1", "RIG:2"]:
    lengths.append(lengths[-1] + len(idx[key]))

KeyError: 'GS'

Move the matrices and create the inputs in a temporary directory.

In [ ]:
system("mkdir scr_int")
system(f"cp scr/data-{name}/hamiltonian_sparse.mtx scr_int/H.mtx")
system(f"cp scr/data-{name}/overlap_sparse.mtx scr_int/S.mtx")
system(f"cp scr/data-{name}/density_kernel_sparse.mtx scr_int/K.mtx")

In [ ]:
from yaml import dump
args = {"regions": lengths,
        "reorder": order,
        "mu": logs["GS"].fermi_level,
        "mu_iter": 15,
        "mu_tol": 1e-3,
        "estep": 0.001,
        "kbt_dm": 0.001,
        "kbt_t": 0.001,
        "ec": 2.0}
with open("scr_int/input.yaml", "w") as ofile:
    dump({"chess": {"libnegf": args}}, ofile)

Now run the single shot driver program.

In [ ]:
system("cd scr_int ; mpirun -np 1 ../test_driv")

Plot the transmission.

In [ ]:
def plot_transmission(fname, eF, emin=None, emax=None):                                                      
    from matplotlib import pyplot as plt                                           
                                                                                   
    x = []                                                                         
    y = []                                                                         
    with open(fname) as ifile:                                                     
        for line in ifile:                                                         
            if line.strip().startswith("#") or not line.strip():                   
                continue                                                           
            split = [float(val) for val in line.split()]                           
            x.append(27.2114 * split[0])                                           
            y.append(split[1])                                                     
                                                                                   
    fig, axs = plt.subplots(1, 1, figsize=(6, 3))                               
    axs.plot(x, y, "o--")                                                       
    axs.set_ylim(-0.1, 1.1 * max(y)) 
    if emin is not None and emax is not None:
        axs.set_xlim(emin, emax)
    axs.axvline(eF, color="k")
    axs.set_xlabel("Energy (eV)", fontsize=12)                                  
    axs.set_ylabel("Transmission", fontsize=12)                                 
    plt.tight_layout() 

In [ ]:
plot_transmission(f"scr_int/tunneling_iter_005.dat", logs["GS"].fermi_level * 27.2114)

We have a data point for each step in the search so you can see what happens without adjusting the chemical potential.

In [ ]:
plot_transmission(f"scr_int/tunneling_iter_001.dat", logs["GS"].fermi_level * 27.2114)

## Self Consistent Inclusion
Now we show an example of running self consistently in the BigDFT loop.

In [ ]:
inp = make_input(True, lengths, order)
logs["NEGF"] = calc.run(sys=sys, input=inp, name=name, run_dir="scr2", skip=False)

Plot the transmission.

In [ ]:
plot_transmission(f"scr2/tunneling_iter_005.dat", logs["GS"].fermi_level * 27.2114)

Plot convergence.

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 3))
axs.plot(logs["GS"].get_intermediate_energies(), 'o--', label="GS Calculation")
axs.plot(logs["NEGF"].get_intermediate_energies(), 'x--', label="NEGF Restart")
axs.legend()